In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
len(v1)

384

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
len(dv)

384

In [6]:
v1.dot(dv)

0.32332402

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [8]:
v2.dot(dv)

0.019730492

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
from tqdm.auto import tqdm

In [12]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [13]:
import numpy as np
X = np.array(vectors)

In [14]:
X.shape

(1350, 384)

In [33]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query) # embed query

In [34]:
# scores = [v_query.dot(X[i]) for i in range(len(X))] -> same code but dot is faster
scores = X.dot(v_query) # compute

In [35]:
idx = np.argmax(scores) # found the best matches
idx, scores[idx]

(2, 0.7629411)

In [24]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [29]:
# top5 = np.argsort(scores)[-5:]
# top5 = top5[::-1]
# top5
top5 = np.argsort(-scores)[:5]

In [30]:
scores[top5]

array([0.7629411 , 0.75793725, 0.7192133 , 0.6536312 , 0.56010014],
      dtype=float32)

In [31]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.75793725
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192133
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

In [32]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [38]:
# computes the dot product between each vector (after filtering) and our query vector.
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)